In [ ]:


import os
import re
import pandas as pd

# --- Instructions ---
# 1. Make sure you have pandas and openpyxl installed:
#    pip install pandas openpyxl
#
# 2. Change the 'folder_path' variable below to the path of the
#    folder containing your .txt files.
#    - Example for Windows: 'C:\\Users\\YourUser\\Documents\\GameLogs'
#    - Example for macOS/Linux: '/Users/YourUser/Documents/GameLogs'
#
# 3. Place this script in a convenient location (it doesn't have to be
#    in the same folder as your text files).
#
# 4. Run the script. It will create a file named 'game_data.xlsx'
#    in the same directory where you run the script.

# --- Configuration ---
# ▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼
# PLEASE UPDATE THIS PATH to the folder with your .txt files
folder_path = os.environ.get("DIPLOMACY_TEXT_DIR", "game_messages")
# ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲

# The name of the output Excel file
output_filename = 'real_game_data_2025.xlsx'

# --- Script Logic (No need to edit below this line) ---

def process_text_files(path):
    """
    Reads all .txt files in a given directory, extracts game IDs from filenames,
    and compiles the content into a list of dictionaries.
    """
    if not os.path.isdir(path):
        print(f"Error: The folder path '{path}' does not exist.")
        print("Please update the 'folder_path' variable in the script.")
        return None

    # This pattern will find one or more digits (\d+) after 'game_'
    # and before '_messages.txt'
    pattern = re.compile(r'game_(\d+)_messages\.txt')
    
    # A list to hold the data for our Excel file
    data_list = []

    print(f"Scanning folder: {path}")

    # Loop through every file in the specified folder
    for filename in os.listdir(path):
        match = pattern.match(filename)
        # Check if the filename matches our 'game_{id}_messages.txt' format
        if match:
            # The game_id is the first captured group from our regex
            game_id = match.group(1)
            
            # Create the full path to the file
            file_path = os.path.join(path, filename)
            
            try:
                # Open and read the entire content of the file
                # 'utf-8' encoding is a common standard, change if you use another
                with open(file_path, 'r', encoding='utf-8') as file:
                    text_content = file.read()
                
                # Append the extracted data as a dictionary to our list
                data_list.append({'game_id': game_id, 'text': text_content})
                print(f"Successfully processed: {filename}")

            except Exception as e:
                print(f"Could not read file {filename}. Error: {e}")

    return data_list

def create_excel_file(data, output_name):
    """
    Converts a list of dictionaries to a pandas DataFrame and saves it
    as an Excel file.
    """
    if not data:
        print("No matching text files were found. No Excel file will be created.")
        return

    # Create a pandas DataFrame from our list of data
    df = pd.DataFrame(data)

    # Reorder columns to have 'game_id' first, then 'text'
    df = df[['game_id', 'text']]

    try:
        # Save the DataFrame to an .xlsx file
        # 'index=False' prevents pandas from writing row numbers into the file
        df.to_excel(output_name, index=False)
        print(f"\n✅ Success! All data has been saved to '{output_name}'")
    except Exception as e:
        print(f"\n❌ Error: Could not save the Excel file. Reason: {e}")


if __name__ == "__main__":
    # Run the main functions
    compiled_data = process_text_files(folder_path)
    if compiled_data:
        create_excel_file(compiled_data, output_filename)


In [ ]:
import pandas as pd
import google.generativeai as genai
import os
import time
import concurrent.futures

# --- Configuration ---
# IMPORTANT: It is strongly recommended to use environment variables for API keys
# rather than hardcoding them in the script.
API_KEY = os.environ.get("GEMINI_API_KEY", "")
MODEL_NAME = "gemini-2.5-flash"
XLSX_PATH = "real_game_data_2025.xlsx"
TEXT_COLUMN = "text"
SUMMARY_COLUMN = "Summary"

# --- Concurrency Setting ---
# Number of rows to process in parallel.
CONCURRENT_WORKERS = 1

# --- Prompt Template ---
PROMPT_TEMPLATE = """Objective: Transform raw, dialogue-based exchanges from the game 'Diplomacy' into a concise, narrative summary, written as if recounting historical events.

Instructions:
Your task is to act as a historian and storyteller. You will receive dialogue between nations and must rewrite it as a flowing, in-universe narrative. Capture the essence of the negotiations, alliances, and tensions, presenting them as actions and decisions made by the countries themselves.

Strictly Adhere to the Following Rules:

1.  **Narrative Form Only**: Convert the dialogue format (e.g., "France: ...", "Germany: ...") into descriptive prose.
2.  **In-Universe Perspective**: Your narrative must be written from a completely in-universe perspective. Treat the countries (England, France, etc.) as the actual actors. Strictly avoid any meta-commentary or words that refer to the game or the communication itself, such as 'players', 'conversation', 'messages', 'dialogue', or 'communication lines'.
3.  **Preserve Core Meaning**: The intentions, proposals, and outcomes must remain perfectly intact. Do not add information or interpretations not present in the original text.
4.  **Strict Word Count**: The original text contains approximately **{word_count} words**. Your final narrative summary must be very close to this length. Strictly adhere to this word count.
5.  **Output Only the Summary**: Your entire response must consist only of the final narrative text. Do not include any introductory phrases, explanations, or concluding remarks.

Style Example:
You must follow the style of the summary provided below precisely-

England reached out to France in the Spring of 1901, proposing a demilitarized zone in the English Channel for the first year. France cautiously agreed to the idea, but sought further assurances, asking what England's naval movements would be. In return for keeping the Channel clear, England requested that France not build a new fleet in Brest, a condition which France accepted, solidifying their initial pact.

Dialogue to Process:
{dialogue_text}
"""

# Global model instance to be shared across threads
model = None

def process_row(index, row):
    """
    Processes a single row of the DataFrame.
    This function is designed to be called concurrently by the thread pool.
    """
    global model

    print(f"-> Starting processing for row {index + 1}...")
    dialogue = row[TEXT_COLUMN]

    if not isinstance(dialogue, str):
        print(f"  - Warning: Text in row {index + 1} is not valid. Skipping.")
        return index, "SKIPPED - INVALID INPUT"

    try:
        word_count = len(dialogue.split())
        full_prompt = PROMPT_TEMPLATE.format(dialogue_text=dialogue, word_count=word_count)
        
        response = model.generate_content(full_prompt)
        cleaned_summary = response.text.strip()
        
        print(f"  - Success: Row {index + 1} summary generated ({len(cleaned_summary.split())} words).")
        return index, cleaned_summary

    except Exception as e:
        print(f"  - ERROR on API call for row {index + 1}: {e}")
        # Adding a small delay in case of API rate limiting errors
        time.sleep(2)
        return index, f"ERROR: {e}"


def main():
    """Main function to configure API, process data concurrently, and save results."""
    global model
    print(f"--- Starting Diplomacy Narrative Generation with {MODEL_NAME} (Concurrent Mode) ---")

    # 1. Configure the Gemini API
    if not API_KEY or "AIza" not in API_KEY:
        print("FATAL ERROR: GOOGLE_API_KEY is not set or is invalid.")
        return
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel(MODEL_NAME)
    print(f"Gemini API configured for model: {MODEL_NAME}")

    # 2. Load and prepare the dataset
    if not os.path.exists(XLSX_PATH):
        print(f"FATAL ERROR: Input file not found at {XLSX_PATH}")
        return
    df = pd.read_excel(XLSX_PATH)
    print(f"Loaded {len(df)} total rows from {XLSX_PATH}.")

    if SUMMARY_COLUMN not in df.columns:
        df[SUMMARY_COLUMN] = ""

    # 3. Identify rows that need processing
    rows_to_process = []
    for index, row in df.iterrows():
        if pd.isna(row[SUMMARY_COLUMN]) or row[SUMMARY_COLUMN].strip() == "":
            rows_to_process.append((index, row))
    
    total_to_process = len(rows_to_process)
    if total_to_process == 0:
        print("--- No rows to process. All summaries are already generated. ---")
        return
        
    print(f"Found {total_to_process} rows needing a summary. Processing with {CONCURRENT_WORKERS} parallel workers.")
    
    processed_count = 0
    start_time = time.time()

    # 4. Process rows concurrently using a ThreadPoolExecutor
    with concurrent.futures.ThreadPoolExecutor(max_workers=CONCURRENT_WORKERS) as executor:
        # Create a dictionary to map futures to their row index
        future_to_index = {executor.submit(process_row, index, row): index for index, row in rows_to_process}

        for future in concurrent.futures.as_completed(future_to_index):
            index = future_to_index[future]
            try:
                # Retrieve the result from the completed future
                result_index, summary = future.result()
                df.loc[result_index, SUMMARY_COLUMN] = summary
            except Exception as exc:
                print(f"  - CRITICAL ERROR processing future for row {index + 1}: {exc}")
                df.loc[index, SUMMARY_COLUMN] = f"CRITICAL ERROR: {exc}"
            
            processed_count += 1

            # 5. Save progress after each batch of workers completes
            if processed_count % CONCURRENT_WORKERS == 0 or processed_count == total_to_process:
                try:
                    df.to_excel(XLSX_PATH, index=False, engine='openpyxl')
                    elapsed_time = time.time() - start_time
                    print(f"--- Progress saved to {XLSX_PATH} ({processed_count}/{total_to_process} completed in {elapsed_time:.2f}s) ---")
                except Exception as e:
                    print(f"  - CRITICAL ERROR saving file: {e}")

    total_time = time.time() - start_time
    print(f"--- All {total_to_process} rows processed in {total_time:.2f} seconds. Final file saved. ---")

if __name__ == "__main__":
    main()
